# Flight Data Prediction with PySpark

## 1. Setup

In [1]:
#Checking the installed Java version
!java -version

openjdk version "17.0.17" 2025-10-21
OpenJDK Runtime Environment (build 17.0.17+10-Ubuntu-124.04)
OpenJDK 64-Bit Server VM (build 17.0.17+10-Ubuntu-124.04, mixed mode, sharing)


In [2]:
!pip install pyspark

In [3]:
# Install Java 17
!sudo apt-get update
!sudo apt-get install -y openjdk-17-jdk-headless

Hit:1 https://download.docker.com/linux/ubuntu noble InRelease
Hit:2 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble InRelease
Hit:3 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:4 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:5 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-security InRelease
Get:6 https://cli.github.com/packages stable InRelease [3917 B]                
Hit:7 https://packages.cloud.google.com/apt cloud-sdk InRelease                
Hit:8 http://deb.wakemeops.com/wakemeops stable InRelease                      
Hit:9 https://security.ubuntu.com/ubuntu noble-security InRelease              
Hit:10 https://archive.ubuntu.com/ubuntu noble InRelease                   
Hit:11 https://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:12 https://archive.ubuntu.com/ubuntu noble-backports InRelease
Fetched 3917 B in 1s (4604 B/s)
Reading package lists... Done
Reading package lists... Done
Building d

In [4]:
# Set JAVA_HOME to Java 17
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

In [5]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
        .master("local[*]")\
        .appName("flights")\
        .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/03 01:17:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/12/03 01:17:26 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [6]:
df = spark.read.csv("/teamspace/studios/this_studio/Project/data/report_2017_2018/part-00000-cf1ad284-3883-452e-a227-e1f19d44f4e0-c000.csv",
                    header=True,
                    inferSchema=True)

In [ ]:
from pyspark.sql.functions import col, count, when

total_rows = df.count()
null_counts = df.select([count(when(col(c).isNull() | (col(c) == ""), c)).alias(c) for c in df.columns])
null_counts_dict = null_counts.collect()[0].asDict()

cols_to_keep = [c for c, null_count in null_counts_dict.items() if null_count / total_rows < 0.8]

df = df.select(cols_to_keep)

df.show(5)

25/12/03 01:18:05 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----+-------+-----+----------+---------+----------+-----------------+------------------------+---------------------------+-----------+-------------------------------+---------------+------------------+------------------+------+--------------+-----------+---------------+---------------+---------+-------------+----------------+----------------+----+---------------+---------+-------------+-------------+-------+----------+-------+--------+---------------+--------+--------------------+----------+-------+---------+--------+------+----------+-------+--------+---------------+--------+------------------+----------+---------+--------+--------------+-----------------+-------+-------+--------+-------------+------------------+
|Year|Quarter|Month|DayofMonth|DayOfWeek|FlightDate|Reporting_Airline|DOT_ID_Reporting_Airline|IATA_CODE_Reporting_Airline|Tail_Number|Flight_Number_Reporting_Airline|OriginAirportID|OriginAirportSeqID|OriginCityMarketID|Origin|OriginCityName|OriginState|OriginStateFips|Orig

In [8]:
df.printSchema()

root
 |-- Year: integer (nullable = true)
 |-- Quarter: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- DayofMonth: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- FlightDate: date (nullable = true)
 |-- Reporting_Airline: string (nullable = true)
 |-- DOT_ID_Reporting_Airline: integer (nullable = true)
 |-- IATA_CODE_Reporting_Airline: string (nullable = true)
 |-- Tail_Number: string (nullable = true)
 |-- Flight_Number_Reporting_Airline: integer (nullable = true)
 |-- OriginAirportID: integer (nullable = true)
 |-- OriginAirportSeqID: integer (nullable = true)
 |-- OriginCityMarketID: integer (nullable = true)
 |-- Origin: string (nullable = true)
 |-- OriginCityName: string (nullable = true)
 |-- OriginState: string (nullable = true)
 |-- OriginStateFips: integer (nullable = true)
 |-- OriginStateName: string (nullable = true)
 |-- OriginWac: integer (nullable = true)
 |-- DestAirportID: integer (nullable = true)
 |-- DestAirportSeqID: 

## 2. RDDs